# Comparing System Solvers

PyOMES has two orthogonal solver axes — see [`0_README.ipynb`](0_README.ipynb)
and [`01_writing_a_custom_solver.ipynb`](01_writing_a_custom_solver.ipynb) for
the full picture. This notebook is about **Axis 2 — `SystemSolver`**: whole-
system orchestration, given a `Simulation` with multiple CVs, inter-CV links,
and controllers.

Five implementations ship with PyOMES:

| Solver | When to choose it |
|---|---|
| `ExplicitEulerSystemSolver` | Default; correct for slow flows (tau_CFL >> dt) and slow controllers. |
| `StrangSplittingSystemSolver` | Free upgrade to 2nd-order splitting; nearly zero extra cost over Euler. |
| `MultirateSystemSolver` | Fast inter-CV circulation; removes the CFL constraint by subcycling link flux. |
| `ImplicitTransportSystemSolver` | Unconditionally stable transport; best for many-CV compartmental models (HPLC, etc.) with no tight controllers. |
| `MonolithicODESolver` | Tight feedback loops (T_c < dt_h), multi-rate digital controllers, co-integrated integral states (PI/PID, observers). |

Two things are demonstrated below:

- **Section 1 — stability and accuracy.** All five solvers remain stable on a
  well-behaved transport problem (tau_CFL >> dt_h) and conserve mass. Accuracy
  differs slightly; wall-time differences give a rough sense of overhead at
  this small scale.
- **Section 2 — controller sub-stepping.** `MonolithicODESolver` fires a
  periodic controller at exact T_c = 0.01 h sub-step boundaries (10x per
  dt_h = 0.1 h macro step) and co-integrates the controller's integral state
  continuously. `ExplicitEulerSystemSolver` fires the same controller once per
  macro step and does not evolve the integral at all.

Model topology: a 1 L reactor recirculating with a 1 L recycle CV at
Q = 2 L/h (tau_CFL = V/Q = 0.5 h >> dt_h = 0.1 h) — pure transport, no
reactions. Substrate S distributes between the two CVs; all solvers converge
to the same steady state.

In [1]:
import time

from PyOMES.core import (
    AdvectiveLink,
    ControlVolume,
    LiquidPhase,
    Simulation,
)
from PyOMES.core.system_solver import (
    ExplicitEulerSystemSolver,
    ImplicitTransportSystemSolver,
    MonolithicODESolver,
    MultirateSystemSolver,
    StrangSplittingSystemSolver,
)
from PyOMES.control.actions import ControlAction
from PyOMES.control.interfaces import ControllerBase

# Model parameters
Q_L_PER_H = 2.0    # recirculation flow rate (L/h)
V_L       = 1.0    # volume of each CV (L)
S_INIT    = 10.0   # initial substrate in reactor (mol)

# tau_CFL = V / Q = 0.5 h >> dt_h = 0.1 h -> all solvers are CFL-stable.
DT_H    = 0.1
TAU_H   = 1.0
N_STEPS = int(TAU_H / DT_H)

# Controller parameters (Section 2 only).
T_C_H = 0.01   # controller update period (h); 10x per macro step

print("Imports and parameters OK")

Imports and parameters OK


In [2]:
def _build(system_solver=None, controllers=None) -> Simulation:
    """Assemble a fresh two-CV simulation for each solver trial."""
    liq_r = LiquidPhase(n_mol={"S": S_INIT}, V_L=V_L, T_K=310.0)
    liq_c = LiquidPhase(n_mol={"S": 0.0},    V_L=V_L, T_K=310.0)
    return Simulation(
        cvs={
            "reactor": ControlVolume(phases={"liquid": liq_r}, label="reactor"),
            "recycle": ControlVolume(phases={"liquid": liq_c}, label="recycle"),
        },
        links=[
            AdvectiveLink(
                _source_cv_key="reactor", _source_phase_key="liquid",
                _sink_cv_key="recycle",   _sink_phase_key="liquid",
                Q_L_per_h=Q_L_PER_H, _label="fwd",
            ),
            AdvectiveLink(
                _source_cv_key="recycle", _source_phase_key="liquid",
                _sink_cv_key="reactor",   _sink_phase_key="liquid",
                Q_L_per_h=Q_L_PER_H, _label="bwd",
            ),
        ],
        controllers=controllers or [],
        system_solver=system_solver,
        label="solver_comparison",
    )

print("_build() defined")

_build() defined


## 1  Solver agreement — stability and accuracy

Run the same transport problem under all five `SystemSolver` implementations
and compare the final substrate distribution against the first solver run
(Euler), plus wall-clock time.

In [3]:
def section1_agreement() -> None:
    """Compare stability and accuracy profile of all five solvers."""
    print("=" * 65)
    print("Section 1: Solver stability and accuracy (no controller)")
    print(f"  reactor <-> recycle, Q = {Q_L_PER_H} L/h, "
          f"tau = {TAU_H} h, dt = {DT_H} h ({N_STEPS} steps)")
    print(f"  Initial: reactor S = {S_INIT:.1f} mol, recycle S = 0.0 mol")
    print()

    solvers = [
        ("ExplicitEuler",     ExplicitEulerSystemSolver()),
        ("StrangSplitting",   StrangSplittingSystemSolver()),
        ("Multirate",         MultirateSystemSolver()),
        ("ImplicitTransport", ImplicitTransportSystemSolver()),
        ("MonolithicODE",     MonolithicODESolver()),
    ]

    fmt = "{:<22}  {:>10}  {:>10}  {:>12}  {:>8}"
    print(fmt.format("Solver", "S_reactor", "S_recycle", "D vs Euler", "ms"))
    print("-" * 65)

    ref_reactor = None
    for name, solver in solvers:
        sim = _build(system_solver=solver)
        t0 = time.perf_counter()
        result = sim.run(tau_h=TAU_H, n_steps=N_STEPS)
        elapsed_ms = (time.perf_counter() - t0) * 1e3

        S_r = result.liquid_mol["reactor"]["S"][-1]
        S_c = result.liquid_mol["recycle"]["S"][-1]

        if ref_reactor is None:
            ref_reactor = S_r
            delta_str = "baseline"
        else:
            delta = abs(S_r - ref_reactor) / (ref_reactor or 1.0)
            delta_str = f"{delta:.2e}"

        print(fmt.format(
            name,
            f"{S_r:.5f}",
            f"{S_c:.5f}",
            delta_str,
            f"{elapsed_ms:.1f}",
        ))

    total = S_INIT  # mol conserved
    print()
    print(f"  Mass conservation: reactor + recycle ~= {total:.1f} mol (all solvers).")
    print(f"  Analytical: S_reactor(1h) ~= 5.092 mol (exact: 5 + 5*exp(-4)).")
    print()

section1_agreement()

Section 1: Solver stability and accuracy (no controller)
  reactor <-> recycle, Q = 2.0 L/h, tau = 1.0 h, dt = 0.1 h (10 steps)
  Initial: reactor S = 10.0 mol, recycle S = 0.0 mol

Solver                   S_reactor   S_recycle    D vs Euler        ms
-----------------------------------------------------------------
ExplicitEuler              5.60680     4.39320      baseline       0.8
StrangSplitting            5.33317     4.66683      4.88e-02       0.4
Multirate                  5.60680     4.39320      0.00e+00       0.3
ImplicitTransport          5.17286     4.82714      7.74e-02    1383.4
MonolithicODE              5.09158     4.90842      9.19e-02    1013.2

  Mass conservation: reactor + recycle ~= 10.0 mol (all solvers).
  Analytical: S_reactor(1h) ~= 5.092 mol (exact: 5 + 5*exp(-4)).



## 2  Controller sub-stepping

A minimal controller with `di/dt = 1.0` (constant), so the expected integral
after time tau is simply tau itself — easy to verify analytically.

- With `ExplicitEulerSystemSolver`: `compute()` fires once per 0.1 h macro
  step; `state_rates()` is never called, so the integral does not evolve.
- With `MonolithicODESolver`: `state_rates()` drives continuous integral
  evolution (di/dt = 1); `compute()` fires at each 0.01 h T_c boundary
  (10x per step); the final integral matches tau as expected.

In [4]:
class _IntegralController(ControllerBase):
    """Minimal controller demonstrating co-integrated differential state."""

    target_cv_key = "reactor"

    def __init__(self):
        self._integral = 0.0
        self.fire_count = 0

    @property
    def update_period_h(self) -> float:
        return T_C_H

    def differential_state(self):
        return {"integral": self._integral}

    def state_rates(self, env, t_h):
        return {"integral": 1.0}

    def set_state(self, state):
        self._integral = state["integral"]

    def compute(self, state, dt_h):
        self.fire_count += 1
        return ControlAction(
            controller_label="integral_ctrl",
            target_cv_key="reactor",
        )


N_STEPS_2 = 5
TAU_H_2   = N_STEPS_2 * DT_H
FIRES_PER_STEP = int(DT_H / T_C_H)


def section2_controller() -> None:
    """MonolithicODE fires at T_c boundaries and co-integrates the state."""
    print("=" * 65)
    print("Section 2: Controller sub-stepping")
    print(f"  Controller update period: T_c = {T_C_H} h")
    print(f"  Macro step: dt = {DT_H} h  =>  {FIRES_PER_STEP}x per step with MonolithicODE")
    print(f"  Run: tau = {TAU_H_2} h ({N_STEPS_2} macro steps)")
    print(f"  Expected integral (di/dt = 1): {TAU_H_2:.3f}")
    print()

    fmt = "{:<22}  {:>12}  {:>13}  {:>10}  {:>10}"
    print(fmt.format(
        "Solver", "fires/step", "total fires", "integral", "expected"
    ))
    print("-" * 65)

    for name, solver in [
        ("ExplicitEuler",  ExplicitEulerSystemSolver()),
        ("MonolithicODE",  MonolithicODESolver()),
    ]:
        ctrl = _IntegralController()
        sim = _build(system_solver=solver, controllers=[ctrl])
        sim.run(tau_h=TAU_H_2, n_steps=N_STEPS_2)

        fires_per_step = ctrl.fire_count / N_STEPS_2
        print(fmt.format(
            name,
            f"{fires_per_step:.0f}",
            str(ctrl.fire_count),
            f"{ctrl._integral:.4f}",
            f"{TAU_H_2:.4f}",
        ))

    print()
    print("  ExplicitEuler  -- fires once per macro step; integral not evolved.")
    print("  MonolithicODE  -- fires at each T_c boundary; integral co-integrated")
    print(f"                   with transport dynamics (di/dt = 1 => integral = {TAU_H_2}).")
    print()
    print("  => Choose MonolithicODE when T_c < dt_h and integral-state accuracy")
    print("    matters: tight pH/DO loops, multi-rate digital controllers, observers.")

section2_controller()

Section 2: Controller sub-stepping
  Controller update period: T_c = 0.01 h
  Macro step: dt = 0.1 h  =>  10x per step with MonolithicODE
  Run: tau = 0.5 h (5 macro steps)
  Expected integral (di/dt = 1): 0.500

Solver                    fires/step    total fires    integral    expected
-----------------------------------------------------------------
ExplicitEuler                      1              5      0.0000      0.5000
MonolithicODE                     10             50      0.5000      0.5000

  ExplicitEuler  -- fires once per macro step; integral not evolved.
  MonolithicODE  -- fires at each T_c boundary; integral co-integrated
                   with transport dynamics (di/dt = 1 => integral = 0.5).

  => Choose MonolithicODE when T_c < dt_h and integral-state accuracy
    matters: tight pH/DO loops, multi-rate digital controllers, observers.
